# Spotify Tracks · Predicción de popularidad con XGBoost
**MLY1101 – Machine Learning · Evaluación Parcial N°2 (Caso C)**

Este notebook sigue las etapas de CRISP-DM y reproduce lo descrito en el `README.md`:
1. Problema de negocio y KPIs
2. Carga de datos y target
3. Análisis exploratorio (EDA)
4. Preparación de datos
5. Modelado con XGBoost (clasificación y regresión)
6. Interpretación de resultados

**Cómo ejecutarlo:** instalar `requirements.txt` y correr todas las celdas en orden (*Run All*).

## 1. Problema de negocio

Las plataformas y sellos musicales deben decidir qué canciones promocionar, pero hoy lo hacen con criterio subjetivo.
**Problema:** clasificar si una canción será popular (`popularity > 70`) usando sus características de audio, duración, si es explícita y su género.
**Uso esperado:** el modelo entrega un ranking de candidatas para apoyar al equipo de promoción, no reemplaza su decisión.

## 2. KPIs

| Tipo | KPI | Qué mide |
|---|---|---|
| Clasificación (principal) | **F1-score** | Equilibrio entre Precision y Recall en la clase "popular" |
| Clasificación | **Precision** | De las canciones que el modelo marca como populares, cuántas realmente lo son |
| Clasificación | **Recall** | De las canciones realmente populares, cuántas logra detectar |
| Clasificación (apoyo) | **PR-AUC** | Calidad del ranking sin depender del umbral |
| Regresión | **MAE / RMSE** | Error promedio (y error con castigo a los grandes) en puntos de popularidad |

No usamos *accuracy*: solo ~4 % de las canciones es popular, así que predecir "nadie es popular" ya daría ~96 % sin ser útil.

In [ ]:
# Librerías del proyecto (si falta alguna: pip install -r requirements.txt)
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score,
                             precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay,
                             mean_absolute_error, mean_squared_error)
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Parámetros globales: semilla (reproducibilidad) y umbral de popularidad
SEED = 42
THRESHOLD_POPULAR = 70

# Rutas del proyecto: el notebook vive en notebooks/, los datos en data/
BASE = Path("..") if Path("../data").exists() else Path(".")
DATA_PATH = BASE / "data" / "Spotify_Tracks_Dataset.csv"
IMG_DIR, MODELS_DIR = BASE / "images", BASE / "models"
IMG_DIR.mkdir(exist_ok=True); MODELS_DIR.mkdir(exist_ok=True)

## 3. Carga de datos y target

Dataset público *Spotify Tracks* con 114.000 filas y 21 columnas (audio, duración, género, popularidad).
El target `is_popular` vale 1 si `popularity > 70` y 0 en otro caso.

In [ ]:
# Carga el CSV y crea el target binario is_popular (1 = popularity > 70)
df = pd.read_csv(DATA_PATH)
df["is_popular"] = (df["popularity"] > THRESHOLD_POPULAR).astype(int)

# Resumen general y desbalance de clases
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"Canciones populares (popularity > {THRESHOLD_POPULAR}): {df['is_popular'].sum():,} ({df['is_popular'].mean():.2%})")
print(f"Baseline 'nadie es popular' -> accuracy = {1 - df['is_popular'].mean():.2%}")
df.head()

In [ ]:
# Gráfico 1: distribución de popularity (con el umbral) y balance de clases del target
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["popularity"], bins=50, ax=ax[0])
ax[0].axvline(THRESHOLD_POPULAR, color="red", ls="--", label=f"umbral = {THRESHOLD_POPULAR}")
ax[0].set_title("Distribución de popularity"); ax[0].legend()
df["is_popular"].value_counts().sort_index().plot.bar(ax=ax[1], color=["#999", "#1DB954"])
ax[1].set_title("Clases del target (0 = no popular, 1 = popular)")
ax[1].set_xticklabels(["No popular", "Popular"], rotation=0)
plt.tight_layout(); plt.savefig(IMG_DIR / "01_target.png", dpi=120); plt.show()

## 4. Análisis exploratorio (EDA)

### 4.1 Calidad de datos
Revisamos nulos, valores en cero que parecen errores y canciones repetidas.
Lo clave: una misma canción puede aparecer en varios géneros (mismo `track_id`), lo que obliga a separar train/test por canción.

In [ ]:
# Calidad de datos: nulos, ceros sospechosos y canciones repetidas
nulos = df.isna().sum()
print("Columnas con nulos:\n", nulos[nulos > 0], "\n")

print("Valores en cero que podrían ser errores o datos faltantes:")
print(f"  popularity == 0     : {(df['popularity'] == 0).sum():,} ({(df['popularity'] == 0).mean():.1%})")
print(f"  tempo == 0          : {(df['tempo'] == 0).sum():,}")
print(f"  time_signature == 0 : {(df['time_signature'] == 0).sum():,}")
print(f"  duration_ms == 0    : {(df['duration_ms'] == 0).sum():,}\n")

# Canciones repetidas (riesgo de fuga de datos si el split fuera aleatorio)
n_dup_rows = df["track_id"].duplicated(keep=False).sum()
print(f"Filas cuyo track_id aparece más de una vez: {n_dup_rows:,} ({n_dup_rows/len(df):.1%})")
print(f"Tracks únicos: {df['track_id'].nunique():,} de {len(df):,} filas")
print(f"Tracks únicos populares: {df.loc[df.is_popular == 1, 'track_id'].nunique():,} vs filas populares: {df['is_popular'].sum():,}")
print(f"Géneros: {df['track_genre'].nunique()} | filas por género (min-max): {df['track_genre'].value_counts().min()}-{df['track_genre'].value_counts().max()}")

### 4.2 Ceros en `popularity` por género
Los ceros se concentran en ciertos géneros (jazz, iranian, romance, soul, latin, country). Es poco probable que sean canciones realmente sin oyentes:
podrían ser datos sin puntuar. Los mantenemos, pero lo declaramos como limitación del target.

In [ ]:
# Gráfico 2: porcentaje de canciones con popularity = 0 en los 15 géneros más afectados
ceros = (df["popularity"] == 0).groupby(df["track_genre"]).mean().sort_values(ascending=False).head(15)
plt.figure(figsize=(8, 5))
ceros.sort_values().plot.barh(color="#c0392b")
plt.xlabel("Proporción de canciones con popularity = 0"); plt.title("Géneros con más ceros en popularity")
plt.tight_layout(); plt.savefig(IMG_DIR / "02_ceros_por_genero.png", dpi=120); plt.show()

### 4.3 Variables de audio vs. popularidad
Las canciones populares son algo más fuertes (`loudness`) y bailables, y menos acústicas e instrumentales.
Pero las diferencias son pequeñas y ninguna correlación (Spearman) supera |0,08|: ninguna variable sola predice bien la popularidad.

In [ ]:
# Gráfico 3: distribución de cada variable de audio, separando populares y no populares
audio_cols = ["danceability", "energy", "loudness", "speechiness", "acousticness",
              "instrumentalness", "liveness", "valence", "tempo"]
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), audio_cols):
    sns.histplot(data=df, x=col, hue="is_popular", stat="density", common_norm=False,
                 element="step", bins=40, palette=["#999", "#1DB954"], ax=ax, legend=False)
    ax.set_title(col)
# Leyenda manual (colores fijos: gris = no popular, verde = popular)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color="#999", label="No popular"), Patch(color="#1DB954", label="Popular")],
           loc="lower center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0.04, 1, 1]); plt.savefig(IMG_DIR / "03_audio_por_clase.png", dpi=120); plt.show()

# Medias por clase para comparar numéricamente
df.groupby("is_popular")[audio_cols].mean().T.round(3)

In [ ]:
# Gráfico 4: correlación de Spearman de cada variable con popularity
num_cols = audio_cols + ["duration_ms", "key", "mode", "time_signature"]
tmp = df[num_cols].copy(); tmp["explicit"] = df["explicit"].astype(int)
corr = tmp.corrwith(df["popularity"], method="spearman").sort_values()
plt.figure(figsize=(8, 5))
corr.plot.barh(color=["#c0392b" if v < 0 else "#1DB954" for v in corr])
plt.xlabel("Correlación de Spearman con popularity"); plt.title("Relación monótona con popularity")
plt.tight_layout(); plt.savefig(IMG_DIR / "04_correlacion.png", dpi=120); plt.show()

### 4.4 Género y contenido explícito
El género es la variable más informativa: `pop` tiene 28 % de canciones populares y 26 de los 114 géneros no tienen ninguna.
Las canciones explícitas son populares el doble de veces (8,5 % vs 3,9 %).

In [ ]:
# Gráfico 5: % de canciones populares por género (top 15) y por contenido explícito
por_genero = df.groupby("track_genre")["is_popular"].mean().sort_values(ascending=False)
print(f"Géneros sin ninguna canción popular: {(por_genero == 0).sum()} de {len(por_genero)}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [2, 1]})
por_genero.head(15).sort_values().plot.barh(ax=ax[0], color="#1DB954")
ax[0].set_xlabel("Proporción de canciones populares"); ax[0].set_title("Top 15 géneros con más canciones populares")
df.groupby("explicit")["is_popular"].mean().rename({False: "No explícita", True: "Explícita"}).plot.bar(ax=ax[1], color=["#999", "#1DB954"])
ax[1].set_ylabel("Proporción de canciones populares"); ax[1].set_xlabel(""); ax[1].set_title("Popularidad según contenido explícito")
ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=0)
plt.tight_layout(); plt.savefig(IMG_DIR / "05_popular_por_genero.png", dpi=120); plt.show()

## 5. Preparación de datos

- **Se descartan** identificadores y texto (`track_id`, nombres, álbum, `artists`): `artists` haría que el modelo aprenda *quién* es famoso, no *qué* hace popular a una canción.
- **Se usan** 9 variables de audio, `duration_ms`, `key`, `mode`, `time_signature`, `explicit` y `track_genre` (categórica nativa de XGBoost, sin escalado ni one-hot).
- **Split 60/20/20** (train/validación/test) agrupado por `track_id` y estratificado por clase, para que una canción no quede en dos conjuntos a la vez.

In [ ]:
# Define variables predictoras (X), targets (clasificación y regresión) y grupos para el split
DROP_COLS = ["Unnamed: 0", "track_id", "track_name", "album_name", "artists"]
TARGET_CLF, TARGET_REG = "is_popular", "popularity"

X = df.drop(columns=DROP_COLS + [TARGET_CLF, TARGET_REG]).copy()
X["explicit"] = X["explicit"].astype(int)               # bool -> 0/1
X["track_genre"] = X["track_genre"].astype("category")  # XGBoost la usa como categórica nativa
y_clf, y_reg = df[TARGET_CLF], df[TARGET_REG]
groups = df["track_id"]

def split_estratificado_por_grupo(y, groups, n_splits, seed=SEED):
    # Toma el primer fold de StratifiedGroupKFold: ~1/n_splits de los datos, sin compartir track_id
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return next(sgkf.split(np.zeros(len(y)), y, groups))

# 80 % train+val / 20 % test
trval_idx, test_idx = split_estratificado_por_grupo(y_clf, groups, n_splits=5)
# Del 80 %: 75 % train / 25 % validación (early stopping y elección de umbral) -> 60/20/20 global
tr_rel, val_rel = split_estratificado_por_grupo(y_clf.iloc[trval_idx], groups.iloc[trval_idx], n_splits=4)
train_idx, val_idx = trval_idx[tr_rel], trval_idx[val_rel]

X_train, X_val, X_test = X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx]
yc_train, yc_val, yc_test = y_clf.iloc[train_idx], y_clf.iloc[val_idx], y_clf.iloc[test_idx]
yr_train, yr_val, yr_test = y_reg.iloc[train_idx], y_reg.iloc[val_idx], y_reg.iloc[test_idx]

# Verificación: ningún track_id compartido entre conjuntos
g_tr, g_val, g_te = (set(groups.iloc[i]) for i in (train_idx, val_idx, test_idx))
assert not (g_tr & g_val) and not (g_tr & g_te) and not (g_val & g_te), "Hay fuga de track_id entre conjuntos"

# Tamaño y % de populares en cada conjunto (deben ser similares)
resumen = pd.DataFrame({
    "filas": [len(X_train), len(X_val), len(X_test)],
    "% populares": [yc_train.mean(), yc_val.mean(), yc_test.mean()],
}, index=["train", "validación", "test"])
resumen["% populares"] = resumen["% populares"].map("{:.2%}".format)
resumen

## 6. Modelado con XGBoost

### 6.1 Clasificación (modelo principal)
XGBoost captura relaciones no lineales entre variables y maneja género como categórica; justo lo que necesitamos, porque ninguna variable de audio explica sola la popularidad.
`scale_pos_weight` compensa el desbalance (~4 % populares) y `early_stopping_rounds` evita el sobreajuste. Los hiperparámetros son un punto de partida, aún sin afinar.

In [ ]:
# Línea base: un modelo que siempre predice "no popular" (referencia mínima a superar)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, yc_train)
print("Línea base (siempre 'no popular') -> F1 en test:", round(f1_score(yc_test, dummy.predict(X_test)), 4))

# Peso de la clase popular = negativos / positivos (compensa el desbalance)
scale_pos_weight = (yc_train == 0).sum() / (yc_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

# Entrena XGBoost; el conjunto de validación decide cuándo detener el entrenamiento
xgb_clf = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    early_stopping_rounds=50,
    tree_method="hist",
    enable_categorical=True,
    random_state=SEED,
    n_jobs=-1,
)
xgb_clf.fit(X_train, yc_train, eval_set=[(X_val, yc_val)], verbose=False)
print("Mejor iteración:", xgb_clf.best_iteration)

**Umbral de decisión.** Con `scale_pos_weight` las probabilidades se desplazan, por lo que 0,5 rara vez es el mejor corte.
Elegimos el umbral que maximiza F1 en validación y lo evaluamos una sola vez en test.

In [ ]:
# Probabilidades de "popular" en validación y test
proba_val = xgb_clf.predict_proba(X_val)[:, 1]
proba_test = xgb_clf.predict_proba(X_test)[:, 1]

# Elige el umbral que maximiza F1 en VALIDACIÓN (nunca en test)
prec, rec, thr = precision_recall_curve(yc_val, proba_val)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = thr[np.argmax(f1s)]
print(f"Umbral óptimo (validación): {best_thr:.3f}  | F1 validación = {f1s.max():.4f}")

def metricas_clf(y_true, proba, umbral):
    # Calcula Precision, Recall, F1 (con el umbral dado) y PR-AUC (independiente del umbral)
    pred = (proba >= umbral).astype(int)
    return {"Umbral": round(umbral, 3),
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred),
            "F1": f1_score(y_true, pred),
            "PR-AUC": average_precision_score(y_true, proba)}

# Tabla de resultados en TEST: umbral estándar (0.5) vs umbral óptimo
tabla = pd.DataFrame({
    "XGBoost (umbral 0.5)": metricas_clf(yc_test, proba_test, 0.5),
    "XGBoost (umbral óptimo)": metricas_clf(yc_test, proba_test, best_thr),
}).T
print(f"Prevalencia en test (PR-AUC de un modelo aleatorio): {yc_test.mean():.4f}")
tabla.round(4)

In [ ]:
# Matriz de confusión (umbral óptimo) y curva Precision-Recall en test
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

pred_opt = (proba_test >= best_thr).astype(int)
ConfusionMatrixDisplay(confusion_matrix(yc_test, pred_opt),
                       display_labels=["No popular", "Popular"]).plot(ax=ax[0], cmap="Greens", colorbar=False)
ax[0].set_title(f"Matriz de confusión en test (umbral = {best_thr:.2f})")

p, r, _ = precision_recall_curve(yc_test, proba_test)
ax[1].plot(r, p, label=f"XGBoost (PR-AUC = {average_precision_score(yc_test, proba_test):.3f})")
ax[1].axhline(yc_test.mean(), color="gray", ls="--", label="Azar (prevalencia)")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Curva Precision-Recall (test)"); ax[1].legend()
plt.tight_layout(); plt.savefig(IMG_DIR / "06_resultados_clasificacion.png", dpi=120); plt.show()

In [ ]:
# Importancia de variables según XGBoost (ganancia promedio, top 12)
imp = pd.Series(xgb_clf.get_booster().get_score(importance_type="gain")).sort_values(ascending=True)
imp.tail(12).plot.barh(figsize=(8, 5), color="#1DB954")
plt.title("XGBoost · importancia de variables (gain, top 12)"); plt.xlabel("Ganancia promedio")
plt.tight_layout(); plt.savefig(IMG_DIR / "07_importancia_variables.png", dpi=120); plt.show()

### 6.2 Regresión complementaria (MAE / RMSE)
Mismo algoritmo y mismos conjuntos, pero prediciendo `popularity` (0–100).
Entrega el error promedio en puntos de popularidad, más fácil de comunicar al negocio.

In [ ]:
# Línea base de regresión: predecir siempre el promedio de popularity
dummy_reg = DummyRegressor(strategy="mean").fit(X_train, yr_train)

# XGBoost de regresión (misma configuración base que el clasificador)
xgb_reg = XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
    eval_metric="rmse", early_stopping_rounds=50, tree_method="hist",
    enable_categorical=True, random_state=SEED, n_jobs=-1,
)
xgb_reg.fit(X_train, yr_train, eval_set=[(X_val, yr_val)], verbose=False)

def metricas_reg(y_true, pred):
    # MAE y RMSE en puntos de popularidad
    return {"MAE": mean_absolute_error(y_true, pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, pred))}

# Tabla de resultados en TEST
tabla_reg = pd.DataFrame({
    "Línea base (promedio)": metricas_reg(yr_test, dummy_reg.predict(X_test)),
    "XGBoost": metricas_reg(yr_test, xgb_reg.predict(X_test)),
}).T
tabla_reg.round(3)

In [ ]:
# Popularidad real vs predicha en test (la diagonal roja sería una predicción perfecta)
pred_reg = np.clip(xgb_reg.predict(X_test), 0, 100)
plt.figure(figsize=(6, 5))
plt.scatter(yr_test, pred_reg, s=4, alpha=0.2)
plt.plot([0, 100], [0, 100], "r--", label="Predicción perfecta")
plt.axhline(THRESHOLD_POPULAR, color="gray", lw=0.8); plt.axvline(THRESHOLD_POPULAR, color="gray", lw=0.8)
plt.xlabel("Popularidad real"); plt.ylabel("Popularidad predicha"); plt.title("XGBoost regresión (test)")
plt.legend(); plt.tight_layout(); plt.savefig(IMG_DIR / "08_regresion_real_vs_predicho.png", dpi=120); plt.show()

In [ ]:
# Guarda los modelos entrenados en models/ (formato JSON nativo de XGBoost)
xgb_clf.save_model(MODELS_DIR / "xgb_clasificador.json")
xgb_reg.save_model(MODELS_DIR / "xgb_regresor.json")
print("Modelos guardados en:", MODELS_DIR.resolve())

## 7. Interpretación de resultados

> **Completar tras ejecutar el notebook**, con los números de las tablas anteriores:
> 1. ¿El F1 y el PR-AUC superan a la línea base y a la prevalencia (~4 %)?
> 2. Qué significan Precision y Recall con el umbral elegido (ej.: "de cada 100 canciones recomendadas, X son realmente populares").
> 3. El MAE/RMSE en puntos de popularidad, comparado con la distancia al umbral de 70.
> 4. Qué variables pesan más y si tienen sentido para el negocio.